# CASMI26: cross-encoder isomer reranker (~3 h)

Trains `XEnc` (spectrum tokens from the frozen fold-safe FPNet + candidate molecular graph) to pick the true structure
among its same-formula isomers, on spectra the FPNet never saw. Code, settings and checkpoints live in
`MyDrive/casmi26/xenc/`. **Run all, keep the tab open.** If it disconnects, Run all again: it resumes.
Needs: L4 GPU runtime, the `KAGGLE_API_TOKEN` secret, the Google Drive prompt.


In [ ]:
# 1. Drive, credentials, dependencies, inputs
import os, sys, json, glob, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:], flush=True)
    if check and r.returncode: raise RuntimeError(cmd)
    return r
sh(f'{sys.executable} -m pip install -q -U kaggle rdkit numba pyarrow')
SIMD = '/content/drive/MyDrive/casmi26/fpsim_full'
OUT = '/content/drive/MyDrive/casmi26/xenc'; os.makedirs(OUT, exist_ok=True)
print('simulated parts on Drive:', len(glob.glob(SIMD + '/*_meta.parquet')))
R = '/content/fp'; os.makedirs(R, exist_ok=True)
def get(ds, dst, f=None):
    if os.path.exists(dst + '/.ok'): return
    sh(f'kaggle datasets download {ds} -p {dst} --unzip' + (f' -f {f}' if f else ''))
    open(dst + '/.ok', 'w').close()
def get_files(ds, dst):
    if os.path.exists(dst + '/.ok'): return
    os.makedirs(dst, exist_ok=True)
    names = [l.split(',')[0] for l in sh(f'kaggle datasets files {ds} --csv --page-size 500').stdout.strip().splitlines()[1:]]
    for n in names:
        if not n.startswith('pilot_part') and not os.path.exists(f'{dst}/{n}'):
            sh(f'kaggle datasets download {ds} -f {n} -p {dst} -q')
            if os.path.exists(f'{dst}/{n}.zip'):
                sh(f'cd {dst} && unzip -oq {n}.zip && rm {n}.zip')
    open(dst + '/.ok', 'w').close()
get('giantwizard/casmi26-jc-fpsim', R + '/in')
get_files('giantwizard/casmi26-jc-fpsim-sim', R + '/in2')
get('ahmedberatozer/casmi26-v2-pool', R + '/pool')
get('ahmedberatozer/casmi26-v4b-models', R + '/v4b')
get('dmitriigluzdov/casmi26-fold-safe-fpnet', R + '/fs', 'fpnet_fold0.pt')
CODE = os.path.dirname(os.path.dirname(glob.glob(R + '/v4b/**/casmi/fpnet.py', recursive=True)[0]))
W = R + '/run/work'; os.makedirs(W, exist_ok=True)
for f in glob.glob(R + '/pool/**/*', recursive=True):
    if os.path.isfile(f) and not os.path.exists(W + '/' + os.path.basename(f)):
        os.symlink(f, W + '/' + os.path.basename(f))
for f in ('spec_off.npy', 'spec_mz.npy', 'spec_it.npy', 'spec_meta.parquet', 'split.parquet'):
    if not os.path.exists(W + '/' + f):
        os.symlink(glob.glob(R + '/in/**/' + f, recursive=True)[0], W + '/' + f)
IN = R + '/in2'
CK0 = glob.glob(R + '/fs/**/fpnet_fold0.pt', recursive=True)[0]


In [ ]:
# 2. embedded training script + held-back evaluation sids (nothing needs to sync up to Drive)
import pandas as pd
TS = '/content/train_xenc.py'
open(TS, 'w').write('"""Train the engine\'s cross-encoder (casmi/xenc.py XEnc) as a same-formula isomer reranker.\n\nThe spectrum side is a frozen FPNet (token states + its f.z score); the candidate side is the molecular graph. Each\ntraining group = one real training spectrum (fold-safe holdout excluded) + its true pool structure + NEG negatives from\nthe same mass window: the HARD hardest by the frozen FPNet\'s f.z, preferring the true structure\'s own formula, plus random\nwindow members. Loss: softmax cross-entropy over the group. XEnc\'s score is a residual on standardised f.z, so at step 0\nit ranks exactly like the FPNet and only has to learn corrections.\nLogged per 100 steps: XEnc group accuracy vs the f.z-only accuracy on the same groups (the gap = what it adds).\n\nusage: train_xenc.py --fp fpnet_fold0.pt --tag NAME [--hours 2] [--device cuda] [--outdir DIR] [--run RUN_DIR]\n"""\nimport argparse, math, os, sys, time\nimport numpy as np\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--code\', default=\'/Users/b/casmi26/inputs/v4b/code\')\nap.add_argument(\'--run\', default=\'/Users/b/casmi26/fp/run\')\nap.add_argument(\'--fp\', required=True)\nap.add_argument(\'--init\', default=\'\')\nap.add_argument(\'--arch\', default=\'256,4,8\', help=\'d,layers,heads\')\nap.add_argument(\'--bs\', type=int, default=32)\nap.add_argument(\'--neg\', type=int, default=15)\nap.add_argument(\'--hard\', type=int, default=12)\nap.add_argument(\'--lr\', type=float, default=2e-4)\nap.add_argument(\'--warmup\', type=int, default=500)\nap.add_argument(\'--steps\', type=int, default=60000)\nap.add_argument(\'--hours\', type=float, default=0.0)\nap.add_argument(\'--merge_p\', type=float, default=0.3)\nap.add_argument(\'--device\', default=\'cuda\')\nap.add_argument(\'--seed\', type=int, default=0)\nap.add_argument(\'--tag\', required=True)\nap.add_argument(\'--outdir\', default=\'/Users/b/casmi26/xenc\')\nap.add_argument(\'--save_every\', type=float, default=1.0)\nap.add_argument(\'--no_fz\', action=\'store_true\', help=\'independent cross-encoder: f.z is NOT an input (zeros, residual weight frozen at 0); the ranker combines it with f.z\')\nap.add_argument(\'--profile\', action=\'store_true\', help=\'print per-phase timings every 10 steps\')\nap.add_argument(\'--holdout\', default=\'hold,np\')\nap.add_argument(\'--train_fold\', default=\'hold\', help="train on this split fold, which the frozen FPNet never saw (\'\' = the FPNet\'s own training spectra)")\nap.add_argument(\'--exclude_sids\', default=\'\', help=\'parquet/npy of sids kept out of training (the harness evaluation half)\')\na = ap.parse_args()\nsys.path.insert(0, a.code)\nos.chdir(a.run)\nimport torch  # noqa: E402\nimport torch.nn.functional as F  # noqa: E402\nfrom casmi import fpnet, traindata, graphs, xenc  # noqa: E402\nfrom casmi.pool import Pool  # noqa: E402\n\nos.makedirs(a.outdir, exist_ok=True)\ndev = a.device\ntorch.manual_seed(a.seed)\nD = traindata.Data(\'cpu\', seed=a.seed, holdout_folds=tuple(f for f in a.holdout.split(\',\') if f))\nrng = np.random.default_rng(a.seed + 11); D.rng = rng\nif a.train_fold:                                   # honest f.z: spectra of structures the frozen FPNet never trained on\n    import pandas as pd\n    excl = set()\n    if a.exclude_sids:\n        excl = set((pd.read_parquet(a.exclude_sids).sid if a.exclude_sids.endswith(\'.parquet\') else np.load(a.exclude_sids)).tolist())\n    ok = (D.fold[D.sid] == a.train_fold) & (D.sid2pid[D.sid] >= 0) & ~np.isin(D.sid, list(excl))\n    D.tr = np.where(ok)[0]\n    key = D.sid * 2 + (D.mode > 0)\n    D.tr_groups = np.array([D.key2g[k] for k in np.unique(key[D.tr])])\n    print(f\'training on fold {a.train_fold!r}: {len(D.tr):,} spectra, {len(np.unique(D.sid[D.tr])):,} structures \'\n          f\'({len(excl):,} sids excluded)\', flush=True)\nP = Pool(\'work\')\nFORM = np.asarray(P.formula)\nPFP = D.PFP.to(dev)\nfp = fpnet.load_model(a.fp, dev).eval()\nfor p_ in fp.parameters():\n    p_.requires_grad_(False)\nif a.init:\n    model = xenc.load_model(a.init, dev).train()\nelse:\n    d, L, H = (int(x) for x in a.arch.split(\',\'))\n    model = xenc.XEnc(d_spec=fp.d, d=d, layers=L, heads=H).to(dev).train()\nif a.no_fz:\n    with torch.no_grad():\n        model.fz_w.zero_()\n    model.fz_w.requires_grad_(False)\nprint(f\'XEnc {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params | spectra {len(D.tr):,}\', flush=True)\n_gcache = {}\n\n\ndef graph(pid):\n    g = _gcache.get(pid)\n    if g is None:\n        g = graphs.mol_tokens(P.smiles[pid])\n        if g is None:\n            g = (np.zeros((1, graphs.N_ATOM_FEAT), np.uint8), np.zeros((1, 1), np.uint8), np.zeros((1, 1), np.uint8))\n        if len(_gcache) < 400_000:\n            _gcache[pid] = g\n    return g\n\n\ndecay = [p for n, p in model.named_parameters() if p.ndim >= 2]; no_decay = [p for n, p in model.named_parameters() if p.ndim < 2]\nopt = torch.optim.AdamW([dict(params=decay, weight_decay=0.01), dict(params=no_decay, weight_decay=0.0)], lr=a.lr, betas=(0.9, 0.98))\namp = dict(device_type=\'cuda\', dtype=fpnet.autocast_dtype(\'cuda\'), enabled=dev == \'cuda\')\nG = 1 + a.neg\nt0 = time.time(); last_save = t0; ema = {}\nfor step in range(1, a.steps + 1):\n    if a.hours and time.time() - t0 > a.hours * 3600:\n        print(\'time budget reached at step\', step, flush=True); break\n    lr = a.lr * min(1.0, step / a.warmup) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * step / a.steps)))\n    for g_ in opt.param_groups:\n        g_[\'lr\'] = lr\n    _T = [time.time()]\n    ids = D.sample_ids(a.bs)\n    items = [D.item(int(i), a.merge_p, True) for i in ids]\n    pos = np.asarray(D.sid2pid[D.sid[ids]], np.int64)\n    win = np.concatenate([D.negatives(pos) for _ in range(2)], 1)          # 126 same-window draws per query\n    _T.append(time.time())\n    args = fpnet.collate(items, dev)\n    with torch.no_grad(), torch.autocast(**amp):\n        sx, spad = fp.encode(*args)\n        z = fp(*args)[0].float()\n    sx = sx.float()\n    _T.append(time.time())\n    groups = np.empty((a.bs, G), np.int64)\n    with torch.no_grad():\n        for b in range(a.bs):\n            c = np.unique(win[b]); c = c[c != pos[b]]\n            fzc = (fpnet.unpack_gpu(PFP[torch.as_tensor(c, device=dev)], D.nbits) @ z[b]).cpu().numpy()\n            same = FORM[c] == FORM[pos[b]]\n            order = np.lexsort((-fzc, ~same))                              # same formula first, then by f.z\n            hard = c[order[:a.hard]]\n            rest = np.setdiff1d(c, hard)\n            rnd = rng.choice(rest, a.neg - len(hard), replace=len(rest) < a.neg - len(hard)) if len(rest) else rng.choice(c, a.neg - len(hard))\n            g_ = np.concatenate([[pos[b]], hard, rnd])\n            if len(g_) < G:\n                g_ = np.concatenate([g_, rng.integers(0, len(P.smiles), G - len(g_))])\n            groups[b] = g_[:G]\n    _T.append(time.time())\n    gt = torch.as_tensor(groups, device=dev)\n    cfp = fpnet.unpack_gpu(PFP[gt.reshape(-1)], D.nbits).reshape(a.bs, G, -1)\n    fz = torch.einsum(\'bgn,bn->bg\', cfp, z)\n    fzs = (fz - fz.mean(1, keepdim=True)) / (fz.std(1, keepdim=True) + 1e-6)\n    atoms, spd, bond, apad = xenc.collate_graphs([graph(int(p_)) for p_ in groups.reshape(-1)], dev)\n    _T.append(time.time())\n    si = torch.arange(a.bs, device=dev).repeat_interleave(G)\n    with torch.autocast(**amp):\n        sc = model(sx[si], spad[si], atoms, spd, bond, apad, (torch.zeros_like(fzs) if a.no_fz else fzs).reshape(-1)).float().reshape(a.bs, G)\n    tgt = torch.zeros(a.bs, dtype=torch.long, device=dev)\n    loss = F.cross_entropy(sc, tgt)\n    opt.zero_grad(set_to_none=True); loss.backward()\n    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()\n    if a.profile:\n        loss.item(); _T.append(time.time())\n        _d = np.diff(_T)\n        _acc = globals().setdefault(\'_PROF\', np.zeros(len(_d))); _acc += _d\n        if step % 10 == 0:\n            print(\'profile s/step: items %.2f | fp encode %.2f | negatives %.2f | graphs+fz %.2f | xenc fwd/bwd %.2f\' % tuple(_acc / 10), flush=True); _acc[:] = 0\n    vals = dict(loss=loss.item(), acc=(sc.argmax(1) == 0).float().mean().item(), acc_fz=(fzs.argmax(1) == 0).float().mean().item(),\n                same=float(np.mean(FORM[groups[:, 1:]] == FORM[groups[:, :1]])))\n    for k, v in vals.items():\n        ema[k] = v if step == 1 else 0.99 * ema[k] + 0.01 * v\n    if step % 100 == 0:\n        print(f\'step {step} lr {lr:.2e} \' + \' \'.join(f\'{k} {v:.4f}\' for k, v in ema.items()) +\n              f\' gain {ema["acc"] - ema["acc_fz"]:+.4f} | {step / (time.time() - t0):.2f} it/s\', flush=True)\n    if a.save_every and time.time() - last_save > a.save_every * 3600:\n        model.eval(); xenc.save_ckpt(os.path.join(a.outdir, f\'{a.tag}_partial.pt\'), model, step, extra=dict(args=vars(a))); model.train()\n        last_save = time.time(); print(\'saved partial at step\', step, flush=True)\nmodel.eval()\nxenc.save_ckpt(os.path.join(a.outdir, f\'{a.tag}.pt\'), model, step, extra=dict(args=vars(a)))\nprint(\'saved\', os.path.join(a.outdir, f\'{a.tag}.pt\'), f\'{(time.time() - t0) / 60:.1f} min\', flush=True)\n')
EX = '/content/np1_evalhalf.parquet'
pd.DataFrame({'sid': [25, 51, 241, 333, 335, 379, 513, 533, 607, 683, 701, 801, 807, 847, 881, 995, 1037, 1191, 1353, 1373, 1395, 1397, 1533, 1561, 1571, 1615, 1621, 1651, 1655, 1669, 1691, 1703, 1747, 1759, 1827, 1829, 1867, 1927, 1929, 1969, 2017, 2047, 2055, 2063, 2069, 2083, 2089, 2223, 2325, 2375, 2445, 61207, 67483, 101709, 108965, 110921, 119185, 149205, 157023, 180145, 185495, 185497, 185503, 185507, 185541, 185591, 185605, 185633, 185829, 185855, 185899, 185951, 186037, 186065, 186067, 186173, 186183, 186195, 186397, 186421, 186483, 186537, 186579, 186595, 186653, 186659, 186713, 186759, 186763, 186769, 186883, 186933, 186935, 186959, 186973, 186995, 187005, 187171, 187215, 187361, 187367, 187403, 187515, 187661, 187803, 187831, 187873, 187905, 187945, 187979, 188023, 188043, 188077, 188085, 188093, 188113, 188115, 188187, 188199, 188215, 188259, 188349, 188371, 188457, 188473, 188611, 188647, 188683, 188725, 188737, 188769, 188773, 188831, 188853, 188921, 188925, 189021, 189023, 189089, 189105, 189117, 189175, 189183, 189291, 189301, 189327, 189347, 189381, 189395, 189443, 189519, 189533, 189537, 189569, 189601, 189703, 189753, 189857, 189885, 189931, 189945, 189985, 190103, 190115, 190153, 190175, 190199, 190205, 190241, 190271, 190335, 190347, 190367, 190473, 190487, 190509, 190559, 190565, 190585, 190639, 190739, 190881, 190925, 190941, 190999, 191211, 191219, 191229, 191239, 191283, 191323, 191421, 191507, 191605, 191807, 191877, 191899, 191919, 191929, 191945, 191963, 191995, 192013, 192023, 192029, 192047, 192051, 192061, 192105, 192129, 192149, 192201, 192253, 192331, 192375, 192475, 192491, 192499, 192513, 192549, 192551, 192635, 192691, 192791, 192795, 192983, 193055, 193065, 193175, 193177, 193191, 193213, 193239, 193263, 193343, 193379, 193467, 193519, 193549, 193587, 193635, 193643, 193647, 193785, 193791, 193799, 193891, 193939, 194005, 194161, 194179, 194211, 194221, 194293, 194297, 194321, 194419, 194437, 194445, 194449, 194581, 194611, 194641, 194643, 194663, 194703, 194717, 194745, 194753, 194775, 194803, 194839, 194841, 194863, 194889, 195125, 195177, 195205, 195257, 195259, 195321, 195327, 195337, 195427, 195429, 195459, 195465, 195569, 195643, 195703, 195785, 195811, 195927, 195937, 195953, 195959, 196033, 196055, 196067, 196101, 196117, 196157, 196179, 196211, 196357, 196377, 196381, 196401, 196415, 196417, 196523, 196537, 196563, 196611, 196651, 196661, 196669, 196829, 196865, 196929, 197039, 197065, 197109, 197111, 197181, 197203, 197205, 197207, 197311, 197351, 197391, 197393, 197427, 197451, 197453, 197467, 197473, 197607, 197807, 197847, 197891, 197921, 197961, 197989, 198033, 198069, 198123, 198141, 198207, 198279, 198305, 198355, 198389, 198423, 198479, 198527, 198547, 198575, 198603, 198651, 198685, 198707, 198717, 198723, 198769, 198787, 198829, 198909, 199017, 199025, 199043, 199057, 199121, 199171, 199209, 199215, 199229, 199261, 199347, 199437, 199455, 199541, 199545, 199577, 199661, 199701, 199703, 199763, 199913, 199929, 199953, 199959, 199993, 200067, 200101, 200181, 200187, 200201, 200223, 200233, 200301, 200315, 200331, 200349, 200387, 200389, 200415, 200485, 200519, 200549, 200579, 200595, 200633, 200641, 200677, 200689, 200697, 200755, 200865, 200899, 200959, 200961, 200973, 201043, 201073, 201105, 201125, 201411, 201511, 201513, 201549, 201585, 201589, 201591, 201651, 201685, 201703, 201731, 201805, 201861, 201873, 201885, 202001, 202041, 202097, 202141, 202201, 202213, 202223, 202229, 202237, 202239, 202309, 202351, 202391, 202425, 202449, 202453, 202459, 202479, 202553, 202687, 202701, 202715, 202719, 202751, 202757, 202761, 202777, 202835, 202957, 203031, 203131, 203181, 203183, 203211, 203213, 203443, 203453, 203471, 203479, 203495, 203515, 203561, 203615, 203619, 203635, 203645, 203697, 203741, 203785, 203831, 203853, 203905, 203949, 203985, 204001, 204067, 204119, 204227, 204287, 204297, 204447, 204533, 204637, 204665, 204733, 204803, 204837, 204865, 204927, 204943, 204959, 205031, 205221, 205259, 205269, 205285, 205307, 205365, 205367, 205397, 205409, 205627, 205665, 205689, 205767, 205835, 205837, 205847, 205851, 205861, 205903, 206057, 206083, 206183, 206233, 206317, 206411, 206425, 206495, 206507, 206551, 206593, 206605, 206723, 206739, 206801, 206857, 206861, 206879, 206915, 206921, 207039, 207055, 207331, 207339, 207345, 207361, 207365, 207567, 207613, 207919, 207947, 208035, 208047, 208105, 208193, 208247, 208303, 208307, 208401, 208411, 208441, 208453, 208491, 208555, 208565, 208617, 208629, 208637, 208695, 208709, 208731, 208733, 208803, 208889, 208917, 208929, 209031, 209057, 209075, 209119, 209123, 209137, 209173, 209207, 209227, 209249, 209259, 209261, 209267, 209465, 209553, 209637, 209641, 209687, 209693, 209809, 209817, 209865, 209921, 209937, 210015, 210039, 210053, 210207, 210215, 210329, 210365, 210377, 210381, 210401, 210417, 210473, 210545, 210579, 210587, 210593, 210633, 210639, 210719, 210783, 210801, 210827, 210879, 210885, 210929, 210947, 210949, 210959, 211029, 211089, 211099, 211109, 211217, 211279, 211315, 211321, 211341, 211499, 211523, 211569, 211667, 211693, 211707, 211755, 211805, 211809, 211933, 211939, 211947, 212023, 212073, 212089, 212103, 212115, 212167, 212205, 212207, 212317, 212331, 212381, 212393, 212413, 212425, 212495, 212505, 212559, 212565, 212637, 212807, 212845, 212849, 212889, 212955, 212991, 213009, 213045, 213049, 213093, 213185, 213321, 213333, 213351, 213429, 213655, 213667, 213695, 213723, 213753, 213779, 213873, 213909, 213993, 214015, 214069, 214075, 214113, 214133, 214187, 214241, 214273, 214283, 214401, 214469, 214525, 214551, 214553, 214581, 214621, 214641, 214675, 214721, 214901, 214919, 214923, 214937, 214943, 214983, 214999, 215005, 215045, 215057, 215217, 215219, 215237, 215239, 215243, 215361, 215419, 215431, 215447, 215601, 215681, 215759, 215789, 215811, 215845, 215933, 215947, 215971, 216029, 216057, 216067, 216133, 216155, 216163, 216203, 216219, 216225, 216299, 216319, 216337, 216425, 216429, 216443, 216515, 216635, 216791, 216873, 216947, 216961, 217019, 217071, 217083, 217105, 217117, 217127, 217129, 217155, 217271, 217293, 217365, 217443, 217477, 217507, 217511, 217533, 217567, 217597, 217599, 217605, 217635, 217671, 217745, 217749, 217779, 217853, 217893, 217907, 217931, 217955, 217985, 217989, 218001, 218031, 218135, 218155, 218189, 218203, 218213, 218261, 218303, 218365, 218409, 218433, 218447, 218597, 218605, 218627, 218687, 218703, 218777, 218839, 218891, 218947, 218951, 218971, 218983, 219023, 219029, 219093, 219169, 219237, 219301, 219351, 219419, 219467, 219537, 219605, 219615, 219623, 219665, 219681, 219697, 219785, 220019, 220033, 220065, 220113, 220129, 220149, 220231, 220235, 220325, 220333, 220363, 220377, 220389, 220405, 220421, 220541, 220635, 220663, 220681, 220821, 220899, 220941, 221029, 221053, 221121, 221125, 221131, 221253, 221283, 221343, 221359, 221435, 221477, 221493, 221497, 221533, 221571, 221573, 221603, 221721, 221765, 221785, 221851, 221857, 221871, 221953, 221965, 221967, 221973, 222001, 222025, 222043, 222057, 222063, 222095, 222103, 222109, 222117, 222153, 222221, 222255, 222311, 222319, 222343, 222367, 222461, 222495, 222503, 222541, 222545, 222565, 222623, 222693, 222713, 222725, 222739, 222817, 222881, 222945, 222965, 223031, 223079, 223165, 223173, 223249, 223325, 223349, 223531, 223537, 223559, 223567, 223569, 223575, 223713, 223729, 223751, 223791, 223967, 224113, 224129, 224203, 224225, 224287, 224289, 224313, 224329, 224335, 224361, 224447, 224449, 224477, 224573, 224603, 224807, 224817, 224863, 224931, 225077, 225081, 225099, 225103, 225109, 225145, 225149, 225169, 225199, 225241, 225299, 225405, 225409, 225449, 225505, 225527, 225553, 225573, 225579, 225599, 225673, 225731, 225783, 225917, 225937, 225943, 225945, 226015, 226043, 226055, 226145, 226181, 226321, 226505, 226515, 226535, 226541, 226577, 226639, 226701, 226715, 226721, 226743, 226759, 226877, 226949, 226987, 226993, 227037, 227123, 227125, 227187, 227285, 227293, 227401, 227575, 227665, 227671, 227679, 227725, 227745, 227747, 227767, 227853, 227879, 227915, 227929, 228023, 228193, 228221, 228227, 228277, 228297, 228309, 228317, 228337, 228353, 228377, 228441, 228451, 228515, 228527, 228571, 228637, 228661, 228669, 228681, 228683, 228703, 228773, 228783, 228791, 228913, 228941, 228977, 228989, 229059, 229129, 229133, 229151, 229207, 229293, 229303, 229325, 229377, 229475, 229497, 229527, 229539, 229547, 229555, 229569, 229585, 229593, 229607, 229613, 229653, 229701, 229703, 229755, 229771, 229815, 229839, 229869, 229907, 229963, 229975, 230025, 230105, 230189, 230289, 230293, 230309, 230333, 230347, 230361, 230381, 230395, 230465, 230477, 230485, 230507, 230555, 230595, 230607, 230619, 230637, 230677, 230713, 230741, 230743, 230753, 230765, 230811, 230863, 230885, 230899, 230911, 230927, 230929, 230935, 230963, 230971, 231031, 231071, 231093, 231105, 231137, 231143, 231145, 231197, 231215, 231237, 231257, 231289, 231297, 231327, 231329, 231365, 231451, 231463, 231501, 231555, 231573, 231591, 231605, 231619, 231631, 231643, 231661, 231671, 231745, 231759, 231777, 231781, 231785, 231793, 231797, 231809, 231825, 231867, 231873, 231887, 231919, 231973, 231979, 231983, 231995, 232017, 232021, 232025, 232099, 232121, 232187, 232195, 232197, 232241, 232257, 232273, 232299, 232305, 232345, 232347, 232393, 232409, 232421, 232431, 232433, 232467, 232481, 232523, 232549, 232551, 232553, 232565, 232571, 232597, 232625, 232633, 232655, 232659, 232669, 232687, 232717, 232727, 232733, 232775, 232799, 232827, 232829, 232841, 232905, 232909, 232919, 232939, 232963, 232989, 233015, 233027, 233037, 233047, 233051, 233069, 233131, 233157, 233173, 233183, 233207, 233209, 233249, 233257, 233273, 233299, 233305, 233307, 233313, 233319, 233347, 233379, 233385, 233399, 233407, 233415, 233419, 233429, 233435, 233443, 233471, 233477, 233485, 233495, 233529, 233583, 233603, 233611, 233623, 233641, 233647, 233651, 233675, 233695, 233735, 233741, 233769, 233777, 233791, 233795, 233801, 233809, 233837, 233851, 233857, 233859, 233893, 233897, 233943, 233951, 233959, 233967, 233973, 234009, 234047, 234059, 234075, 234113, 234215, 234225, 234241, 234253, 234297, 234307, 234327, 234337, 234343, 234365, 234367, 234381, 234385, 234409, 234411, 234417, 234419, 234431, 234459, 234469, 234497, 234513, 234515, 234527, 234533, 234537, 234585, 234587, 234599, 234607, 234641, 234645, 234661, 234667, 234683, 234733, 234735, 234741, 234769, 234773, 234781, 234801, 234803, 234893, 234905, 234921, 234979, 234989, 234993, 235047, 235079, 235083, 235093, 235133, 235329, 235343, 235345, 235373, 235383, 235417, 235431, 235457, 235477, 235479, 235507, 235541, 235577, 235583, 235613, 235617, 235625, 235629, 235693, 235701, 235731, 235735, 235755, 235841, 235849, 235855, 235879, 235881, 236925, 237025, 237531, 237583, 238087, 238181, 238185, 238207, 238227, 238577, 238603, 239371, 239527, 239797, 240817, 241025, 241063, 242021, 242079, 242311, 242693, 243189, 243489, 243549, 243587, 244875, 245423, 245609, 245663, 245751, 246121, 246137, 246725, 246983, 247287, 247289, 247463, 247851, 247871, 247981, 248433, 248721, 248857, 248879, 248917, 249087, 249189, 249749, 249865, 250245, 251479, 252043, 252159, 252303, 252335, 252415, 253191, 253311, 253457, 254221, 254623, 254795, 254895, 255013, 255049, 255439, 256099, 256573, 256607, 256715, 256777, 256797, 256825, 257063, 257575, 257847, 258477, 258625, 259183, 260551, 260687, 260741, 260889, 261051, 261073, 261121, 261227, 261729, 261779, 262339, 262553, 262775, 263765, 264103, 264143, 264383, 264585, 264967, 264975, 265023, 265025, 265041, 265057, 265063, 265119, 265121, 265163, 265165, 265179, 265189, 265211, 265235, 265271, 265287, 265301, 265313, 265363, 265395, 265401, 265435, 265437, 265465, 265469, 265489, 265549, 265553, 265581, 265585, 265613, 265695, 265713, 265715, 265789, 265827, 265865, 265921, 265941, 265949, 265951, 265983, 266007, 266009, 266047, 266069, 266125, 266207, 266217, 266239, 266241, 266255, 266271, 266287, 266341, 266445, 266495, 266547, 266569, 266575, 266589, 266649, 266743, 266751, 266783, 266813, 266831, 266843, 266985, 267087, 267127, 267197, 267209, 267273, 267281, 267285, 267291, 267309, 267351, 267373, 267405, 267425, 267545, 267549, 267603, 267635, 267675, 267697, 267709, 267727, 267743, 267807, 267813, 267835, 267853, 267887, 267923, 267989, 268075, 268113, 268197, 268239, 268319, 268329, 268365, 268375, 268383, 268451, 268471, 268475, 268477, 268511, 268579, 268635, 268663, 268705, 268709, 268725, 268729, 268735, 268757, 268809, 268831, 268855, 268881, 268905, 268923, 268935, 268939, 268941, 268943, 269031, 269039, 269041, 269063, 269087, 269105, 269107, 269165, 269173, 269185, 269199, 269203, 269275, 269277, 269285, 269317, 269325, 269405, 269419, 269513, 269543, 269619, 269669, 269749, 269879, 269925, 269927, 269953, 269955, 270049, 270053, 270125, 270131, 270149, 270159, 270169, 270207, 270241, 270251, 270259, 270281, 270291, 270379, 270403, 270409, 270415, 270473, 270527, 270545, 270565, 270605, 270645, 270651, 270675, 270719, 270803, 270837, 270937, 270947, 271015, 271039, 271083, 271103, 271119, 271169, 271195, 271201, 271255, 271261, 271287, 271327, 271455, 271509, 271513, 271575, 271605, 271609, 271613, 271615, 271739, 271763, 271829, 271863, 271951, 272001, 272179, 272225, 272287, 272411, 272505, 272529, 272569, 272597, 272629, 272637, 272667, 272713, 272767, 272789, 272793, 272805, 272907, 272919, 272941, 272975, 272979, 272991, 272995, 273069, 273073, 273095, 273103, 273109, 273163, 273187, 273225, 273245, 273353, 273487, 273693, 273713, 273759, 273781, 273795, 273829, 274017, 274097, 274107, 274123, 274125, 274147, 274161, 274177, 274313, 274317, 274367, 274397, 274427, 274431, 274437, 274515, 274545, 274553, 274665, 274673, 274675, 274683, 274685, 274697, 274733, 274743, 274751, 274771, 274929, 274933, 275009, 275033, 275043, 275051, 275071, 275115, 275143, 275175, 275191, 275321, 275349, 275377, 275395, 275439, 275451, 275601, 275627, 275633, 275687, 275713, 275773, 275789, 275795]}).to_parquet(EX)
print('wrote', TS, 'and', EX)


In [ ]:
# 3. train in the background; the log is on Drive (train_xenc.log)
import time
HOURS, BS, TAG = 3.0, 16, 'xenc_nofz'
part = f'{OUT}/{TAG}_partial.pt'
init = f'--init {part}' if os.path.exists(part) else ''
cmd = (f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {sys.executable} {TS} --code {CODE} --run {R}/run --fp {CK0} {init} --device cuda --bs {BS} '
       f'--steps 60000 --hours {HOURS} --exclude_sids {EX} --no_fz --profile --save_every 0.5 --tag {TAG} '
       f'--outdir {OUT} >> {OUT}/train_{TAG}.log 2>&1')
print(cmd)
proc = subprocess.Popen(cmd, shell=True)
while proc.poll() is None:
    time.sleep(300)
    print(open(f'{OUT}/train_{TAG}.log').read().strip().splitlines()[-1], flush=True)
print(open(f'{OUT}/train_{TAG}.log').read()[-2000:])
print('final checkpoint:', os.path.exists(f'{OUT}/{TAG}.pt'))
